# Voice deepfake research audio analysis

This notebook creates reproducible publication figures from a CSV with `location,label`, where `0` is human speech and `1` is AI speech. It does not train a detector or claim experimental performance.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

CSV_PATH = Path(os.environ.get('VOICE_DATA_CSV', 'data/manifest.csv'))
OUTPUT_DIR = Path('artifacts/research_figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SR = 16_000
N_FFT = 512
HOP = 160
N_MELS = 80
df = pd.read_csv(CSV_PATH)
assert {'location', 'label'} <= set(df.columns)
assert set(df['label'].unique()) <= {0, 1}
df['class_name'] = df['label'].map({0: 'Human', 1: 'AI'})
df.head()

In [ ]:
ax = df['class_name'].value_counts().reindex(['Human', 'AI']).plot(kind='bar', color=['#4472c4', '#ed7d31'], figsize=(6, 4))
ax.set_title('Dataset class distribution')
ax.set_ylabel('Number of clips')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=300)
plt.show()

In [ ]:
def load_clip(path):
    audio, rate = librosa.load(path, sr=SR, mono=True, duration=3.0)
    audio = librosa.util.fix_length(audio, size=3 * SR)
    return audio.astype('float32'), rate

def plot_clip(row, output_name):
    audio, rate = load_clip(row['location'])
    mel = librosa.feature.melspectrogram(y=audio, sr=rate, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=2.0)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
    librosa.display.waveshow(audio, sr=rate, ax=axes[0], color='#4472c4')
    axes[0].set_title(f"{row['class_name']} waveform")
    image = librosa.display.specshow(mel_db, sr=rate, hop_length=HOP, x_axis='time', y_axis='mel', ax=axes[1], cmap='magma')
    axes[1].set_title(f"{row['class_name']} log-Mel spectrogram")
    fig.colorbar(image, ax=axes[1], format='%+2.0f dB')
    fig.savefig(OUTPUT_DIR / output_name, dpi=300)
    plt.show()

for label, name in [(0, 'human_example'), (1, 'ai_example')]:
    matches = df[df.label == label]
    if not matches.empty and Path(matches.iloc[0]['location']).exists(): plot_clip(matches.iloc[0], f'{name}_mel.png')

In [ ]:
# Optional aggregate comparison: mean log-Mel energy by class.
class_mels = {}
for label, group in df.groupby('label'):
    matrices = []
    for _, row in group.head(100).iterrows():
        if Path(row['location']).exists():
            audio, rate = load_clip(row['location'])
            matrices.append(librosa.power_to_db(librosa.feature.melspectrogram(y=audio, sr=rate, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS), ref=np.max))
    if matrices: class_mels[label] = np.mean(matrices, axis=0)
if len(class_mels) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    for axis, label in zip(axes, [0, 1]):
        image = librosa.display.specshow(class_mels[label], sr=SR, hop_length=HOP, x_axis='time', y_axis='mel', ax=axis, cmap='magma')
        axis.set_title(['Human', 'AI'][label] + ' mean log-Mel')
    fig.colorbar(image, ax=axes.ravel().tolist(), format='%+2.0f dB')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'class_mean_mel_comparison.png', dpi=300); plt.show()